In [4]:
import torch
import torch.nn as nn 
import torch.nn.functional as F # F

class BaselineCNN(nn.Module): #nn.module sınıfından kalıtım alarak yeni bir sınıf oluşturuyoruz
    def __init__(self, num_classes=38):
        super(BaselineCNN, self).__init__() #nn.module sınıfının __init__ metodunu çağırarak sınıfımızı başlatıyoruz
        # Katman tanımlamaları
        #neden 224x224? Çünkü genellikle görüntü sınıflandırma modelleri, giriş görüntülerini belirli bir boyuta (örneğin 224x224) yeniden boyutlandırarak çalışır. Bu, modelin sabit bir giriş boyutuna sahip olmasını sağlar ve eğitim sürecini kolaylaştır
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1) #şimdi bu 3x3lük bir filter. 16 adet filter var, her bir filter 3 kanallı 
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64 * 28 * 28, 512) #50176 giriş, 512 neurona giden bir fully connected layer sonuc olarak da 512 çıkış verir
        self.fc2 = nn.Linear(512, num_classes) # 38 sınıf için 512 giriş, 38 çıkış verir.  Her çıkış, bir sınıfın olasılığını temsil eder? neden softmax kullanmıyoruz? Çünkü CrossEntropyLoss fonksiyonu, modelin çıktısını doğrudan logitler (yani softmax uygulanmamış değerler) olarak bekler. Bu nedenle, modelin son katmanında softmax uygulamak yerine, doğrudan lineer bir katman kullanarak logitler üretmek daha uygundur. CrossEntropyLoss, bu logitleri alır ve hem softmax uygulamasını hem de kayıp hesaplamasını tek adımda gerçekleştirir. Bu, eğitim sürecini daha verimli hale getirir ve sayısal kararlılığı artırır.
        self.dropout = nn.Dropout(0.5) # 0.5 oranında dropout demek 19 nöronun rastgele kapatılması demek. Bu strateji, modelin belirli nöronlara aşırı bağımlı hale gelmesini önleyerek genelleme yeteneğini artırır. Eğitim sırasında, her iterasyonda RASTGELE SEÇİLEN nöronlar kapatılır (yani, bu nöronların çıktısı sıfırlanır), bu da modelin farklı nöron kombinasyonlarıyla öğrenmesini sağlar ve böylece overfitting riskini azaltır. SÜPER

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = torch.flatten(x, 1) #buranın sonunda xin sizeı ne olur? en başta 224x224x3 ise conv1'den sonra 224x224x16 olur, conv2'den sonra 112x112x32 olur, conv3'ten sonra 56x56x64 olur. flatten işleminden sonra ise 64*28*28 olur yani 50176 olur
        x = F.relu(self.fc1(x))
        x = self.dropout(x) #512 nöronun yarısını rastgele kapatır, 
        return self.fc2(x)

print("Model mimarisi başarıyla tanımlandı!")

Model mimarisi başarıyla tanımlandı!


In [5]:
# Cihaz kontrolü
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #cuda gpuların kullanılabilir olup olmadığını kontrol eder. Eğer cuda destekli bir GPU varsa, model ve veriler GPU'ya taşınır, aksi takdirde CPU kullanılır. 
print(f"Kullanılan Cihaz: {device}")

# Modeli oluştur ve ekran kartına gönder
model = BaselineCNN().to(device)

# Sahte bir veri (batch_size=1, 3 kanal, 224x224 boyut) oluşturup test edelim
dummy_input = torch.randn(1, 3, 224, 224).to(device) #batch_size nedir, tekte paralel işleyebileceği veri adedi.
output = model(dummy_input)

print(f"Model Çıkış Boyutu: {output.shape}") # [1, 38] görmeliyiz
if output.shape == (1, 38):
    print("Mükemmel! Model sıfırdan sorunsuz çalışıyor.")

Kullanılan Cihaz: cuda
Model Çıkış Boyutu: torch.Size([1, 38])
Mükemmel! Model sıfırdan sorunsuz çalışıyor.


## Dataset Preprocessing & Data Augmentation

**Normalization:** Dataset-specific mean ve std değerlerini hesaplıyoruz. Tüm eğitim görsellerini tarayarak kendi datasetimize özel normalizasyon değerleri elde ediyoruz. Bu, "from scratch" yaklaşımımızla tutarlıdır.

**Train Augmentation Teknikleri:**
- `RandomResizedCrop(224)` — Görselin rastgele bir bölgesini kırpıp 224x224'e yeniden boyutlandırır. Farklı zoom seviyeleri sağlar.
- `RandomHorizontalFlip()` — %50 olasılıkla yatay çevirme. Yapraklar her iki yönde de olabilir.
- `RandomVerticalFlip()` — %30 olasılıkla dikey çevirme.
- `RandomRotation(15)` — ±15 derece rastgele döndürme.
- `ColorJitter` — Parlaklık, kontrast, doygunluk ve renk tonunda rastgele değişiklik. Farklı ışık koşullarını simüle eder.
- `GaussianBlur` — Hafif bulanıklaştırma, gerçek dünya koşullarını simüle eder.

**Validation/Test:** Augmentation UYGULANMAZ. Sadece resize + normalize. Çünkü değerlendirme sırasında tutarlı sonuçlar istiyoruz.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ============================================================
# Dataset yolu
# ============================================================
data_dir = 'C:/Users/ugurd/Desktop/Plant_Disease_Dataset'

# ============================================================
# Dataset-specific mean & std hesaplama
# ============================================================
# Önce sadece ToTensor ile yükleyip tüm eğitim verisi üzerinden
# kanal bazında ortalama ve standart sapma hesaplıyoruz.
# Bu, proposal'daki "compute dataset-specific mean and std" gereksinimini karşılar.

print("Dataset-specific mean ve std hesaplanıyor...")
print("(Bu işlem tüm eğitim verisini taradığı için birkaç dakika sürebilir)\n")

# Geçici olarak sadece ToTensor uygulayarak veriyi yükle (0-1 aralığına dönüşür)
temp_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor() #PIL görüntüsünü PyTorch tensörüne dönüştürür ve piksel değerlerini [0, 1] aralığına sıkıştırır, pytorch tensörlerle çalışır. PIL açılımı Python Imaging Library'dir, görüntü işleme için kullanılan bir kütüphanedir.
    #PIL, Python'da görüntüleri açmak, düzenlemek ve kaydetmek için yaygın olarak kullanılır. Ancak, PyTorch ile çalışırken, görüntüleri tensörlere dönüştürmek gerekir, bu nedenle ToTensor() dönüşümü kullanılır. 
    #ToTensor(), PIL görüntüsünü PyTorch tensörüne dönüştürür ve piksel değerlerini [0, 1] aralığına sıkıştırır. Bu sayede, model eğitimi sırasında daha stabil ve hızlı bir şekilde çalışabiliriz.
])

temp_dataset = datasets.ImageFolder(root=f'{data_dir}/train', transform=temp_transform) #image, label= temp_dataset[i] şeklinde erişebiliriz. Burada image bir Pytorch tensörüdür ve label ise o görüntünün sınıf indeksidir. neğin, class1: 0, class2:
temp_loader = DataLoader(temp_dataset, batch_size=64, shuffle=False, num_workers=4) 

# Kanal bazında (R, G, B) ortalama ve std hesapla
mean = torch.zeros(3)
std = torch.zeros(3)
n_pixels = 0

for images, _ in temp_loader:
    batch_size = images.size(0)
    # images shape: [B, C, H, W] -> [B, C, H*W]
    images = images.view(batch_size, 3, -1)
    mean += images.mean(dim=2).sum(dim=0)
    std += images.std(dim=2).sum(dim=0)
    n_pixels += batch_size

mean /= n_pixels
std /= n_pixels

dataset_mean = mean.tolist()
dataset_std = std.tolist()

print(f"Dataset Mean (R, G, B): {[f'{v:.4f}' for v in dataset_mean]}")
print(f"Dataset Std  (R, G, B): {[f'{v:.4f}' for v in dataset_std]}")
print(f"(Toplam {n_pixels} görsel üzerinden hesaplandı)\n")

# Geçici dataset ve loader'ı temizle
del temp_dataset, temp_loader

# ============================================================
# Transform tanımlamaları
# ============================================================

# TRAIN: Data Augmentation + Normalization
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),  # orijinal boyutun %80-100 arasında rastgele kırpmma yapacağız önce, ne kadar kırpacağımız da random, nereyi kırpacağımız da random. Daha sonra bu kırpılan parçayı 224x224 boyutuna yeniden boyutlandıracağız.
    transforms.RandomHorizontalFlip(),                    # Horizontal flip: Maalesef yanıltıcı bir isimlendirme, dikey bir ayna var gibi düşün.
    transforms.RandomVerticalFlip(p=0.3),                  # %30 olasılıkla Vertical flip yani yatay bir ayna var gibi düşün. Yani yaprakları alt üst çevirebiliriz.
    transforms.RandomRotation(15),                         # ±15 derece rastgele döndürme
    transforms.ColorJitter(
        brightness=0.2,   # parlaklık ±%20 aynen aralık ama parlaklık için %20 kadar artırabilir veya azaltabiliriz
        contrast=0.2,     # kontrast ±%20
        saturation=0.2,   # doygunluk ±%20
        hue=0.05          # renk tonu ±%5 (yaprak renkleri çok fazla kaymamalı)
    ),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),  # hafif bulanıklaştırma
    transforms.ToTensor(),                                      # PIL -> Tensor [0,1] aralığına dönüştürür
    transforms.Normalize(mean=dataset_mean, std=dataset_std)    # dataset-specific normalizasyon
])

# VALIDATION & TEST: Sadece resize + normalize (augmentation YOK)
eval_transform = transforms.Compose([
    transforms.Resize(256),        # önce 256'ya küçült
    transforms.CenterCrop(224),    # ortadan 224x224 kırp (kenar bozulmalarını önler)
    transforms.ToTensor(),
    transforms.Normalize(mean=dataset_mean, std=dataset_std)  # aynı dataset-specific normalizasyon
])

# ============================================================
# Dataset yükleme
# ============================================================
train_dataset = datasets.ImageFolder(root=f'{data_dir}/train', transform=train_transform) 
#train_dataset = 70K görselin yollarını tutan bir liste+ her erişimde transform uygulayan bir mekanizma

#train_dataset[0]  → diskten oku → transform uygula → tensör döndür (bellekte tutmaz)
#train_dataset[0]  → diskten oku → transform uygula → farklı tensör döndür
#yani böyle bellekte 70k görsel tutmuyor, her erişimde diskten okuyor ve transform uyguluyor. Bu da bellek kullanımını azaltır ve farklı augmentasyonlar sağlar.

valid_dataset = datasets.ImageFolder(root=f'{data_dir}/valid', transform=eval_transform)# bu da aynı, bu valid_dataset objesi, çağrıldığında tanımlanmış olduğu eval_transform'ı uygulayarak görüntüleri yükler ve dönüştürür. Yani valid_dataset[0] dediğimizde, diskten o görüntüyü okur, eval_transform'ı uygular ve bize bir tensör döndürür.
#Bu sayede validasyon verisi için de aynı normalizasyon ve boyutlandırma işlemleri uygulanmış olur.

# ============================================================
# Sınıf isimleri ve sayıları
# ============================================================
class_names = train_dataset.classes
num_classes = len(class_names)
print(f"Toplam sınıf sayısı: {num_classes}")
print(f"Eğitim verisi: {len(train_dataset)} görsel")
print(f"Validasyon verisi: {len(valid_dataset)} görsel")
print(f"\nİlk 10 sınıf:")
for i, name in enumerate(class_names[:10]):
    print(f"  {i}: {name}")

Dataset-specific mean ve std hesaplanıyor...
(Bu işlem tüm eğitim verisini taradığı için birkaç dakika sürebilir)

Dataset Mean (R, G, B): ['0.4760', '0.5004', '0.4266']
Dataset Std  (R, G, B): ['0.1776', '0.1510', '0.1962']
(Toplam 70295 görsel üzerinden hesaplandı)

Toplam sınıf sayısı: 38
Eğitim verisi: 70295 görsel
Validasyon verisi: 17572 görsel

İlk 10 sınıf:
  0: Apple___Apple_scab
  1: Apple___Black_rot
  2: Apple___Cedar_apple_rust
  3: Apple___healthy
  4: Blueberry___healthy
  5: Cherry_(including_sour)___Powdery_mildew
  6: Cherry_(including_sour)___healthy
  7: Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
  8: Corn_(maize)___Common_rust_
  9: Corn_(maize)___Northern_Leaf_Blight


In [ ]:
# ============================================================
# DataLoader oluşturma
# ============================================================
# batch_size=32: Her iterasyonda 32 görsel paralel işlenir
# shuffle=True: Eğitim verisini her epoch'ta karıştırır (overfitting'i azaltır)
# num_workers=4: 4 paralel işlem ile veri yükleme hızlandırılır
# pin_memory=True: GPU'ya veri transferini hızlandırır

BATCH_SIZE = 32
#Train_Set ile elde ettiklerimizi(transform uyguladık) daha efficient bir şekilde modelimize verebilmek için DataLoader kullanıyoruz. DataLoader, veriyi belirli bir batch_size'a göre böler ve her erişimde bu batch'leri oluşturur. shuffle=True ile eğitim verisini her epoch'ta karıştırarak modelin farklı veri kombinasyonlarıyla öğrenmesini sağlıyoruz, bu da overfitting riskini azaltır.
#num_workers=4 ile veri yükleme işlemini hızlandırmak için 4 paralel işlem kullanıyoruz. pin_memory=True ise GPU'ya veri transferini hızlandırır, çünkü veriler CPU belleğinde tutulurken GPU'ya daha hızlı aktarılabilir hale gelir.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                          num_workers=4, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                          num_workers=4, pin_memory=True)

print(f"Train batch sayısı: {len(train_loader)} (her biri {BATCH_SIZE} görsel)")
print(f"Valid batch sayısı: {len(valid_loader)} (her biri {BATCH_SIZE} görsel)")

# İlk batch'i kontrol edelim
images, labels = next(iter(train_loader))
print(f"\nBir batch'in boyutu: {images.shape}")  # [32, 3, 224, 224] görmeliyiz
print(f"Label'ların boyutu: {labels.shape}")      # [32] dir çünkü her bir tensörde 31 farklı veri var, her birisinin sınıfını tutmak için 32lik liste ihtiyaç var

Train batch sayısı: 2197 (her biri 32 görsel)
Valid batch sayısı: 550 (her biri 32 görsel)

Bir batch'in boyutu: torch.Size([32, 3, 224, 224])
Label'ların boyutu: torch.Size([32])


: 

## Eğitim (Training Loop)

Aşağıda modeli eğitmek için gerekli bileşenler:

- **Loss Function:** `CrossEntropyLoss` — çok sınıflı sınıflandırma için standart. İçinde softmax + negative log-likelihood birleşik.
- **Optimizer:** `Adam` — adaptive learning rate, SGD'den daha hızlı yakınsar.
- **Scheduler:** `ReduceLROnPlateau` — validation loss iyileşmezse learning rate'i otomatik düşürür.
- **Early Stopping:** Validation loss belirli epoch boyunca iyileşmezse eğitimi durdurur (overfitting'i önler).

In [ ]:
import torch.optim as optim
import time

# ============================================================
# Modeli yeniden oluştur (augmentation ile uyumlu)
# ============================================================
model = BaselineCNN(num_classes=num_classes).to(device)

# ============================================================
# Loss, Optimizer, Scheduler
# ============================================================
criterion = nn.CrossEntropyLoss()  # çok sınıflı sınıflandırma için

optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer, lr=0.001 başlangıç öğrenme hızı

# Validation loss 5 epoch boyunca iyileşmezse lr'yi 0.1 ile çarp (lr = lr * 0.1)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, verbose=True)

# ============================================================
# Eğitim parametreleri
# ============================================================
NUM_EPOCHS = 30          # maksimum epoch sayısı
EARLY_STOP_PATIENCE = 7  # 7 epoch boyunca iyileşme olmazsa dur

print("Eğitim hazırlığı tamamlandı!")
print(f"  - Optimizer: Adam (lr=0.001)")
print(f"  - Loss: CrossEntropyLoss")
print(f"  - Scheduler: ReduceLROnPlateau (patience=5)")
print(f"  - Max Epoch: {NUM_EPOCHS}")
print(f"  - Early Stopping Patience: {EARLY_STOP_PATIENCE}")

In [ ]:
# ============================================================
# Eğitim Döngüsü
# ============================================================

# Sonuçları takip etmek için listeler
train_losses = []
valid_losses = []
train_accuracies = []
valid_accuracies = []

best_valid_loss = float('inf')
epochs_without_improvement = 0
best_model_state = None

for epoch in range(NUM_EPOCHS):
    start_time = time.time()
    
    # -------------------- TRAIN AŞAMASI --------------------
    model.train()  # dropout aktif, batch norm eğitim modunda
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()       # gradyanları sıfırla
        outputs = model(images)     # forward pass
        loss = criterion(outputs, labels)  # loss hesapla
        loss.backward()             # backward pass (gradyanları hesapla)
        optimizer.step()            # ağırlıkları güncelle
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)  # en yüksek olasılıklı sınıfı al
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # Her 100 batch'te bir ilerleme göster
        if (batch_idx + 1) % 100 == 0:
            print(f"  Epoch [{epoch+1}/{NUM_EPOCHS}] Batch [{batch_idx+1}/{len(train_loader)}] "
                  f"Loss: {loss.item():.4f}")
    
    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    
    # -------------------- VALIDATION AŞAMASI --------------------
    model.eval()  # dropout kapalı, batch norm eval modunda
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():  # gradyan hesaplamayı kapat (bellek tasarrufu + hız)
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    valid_loss = running_loss / len(valid_loader)
    valid_acc = 100 * correct / total
    valid_losses.append(valid_loss)
    valid_accuracies.append(valid_acc)
    
    # Learning rate scheduler güncelle
    scheduler.step(valid_loss)
    
    elapsed = time.time() - start_time
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}] - {elapsed:.1f}s")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Valid Loss: {valid_loss:.4f} | Valid Acc: {valid_acc:.2f}%")
    print(f"  Learning Rate: {current_lr:.6f}")
    
    # -------------------- EARLY STOPPING --------------------
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        epochs_without_improvement = 0
        best_model_state = model.state_dict().copy()  # en iyi modeli kaydet
        print(f"  >> En iyi model güncellendi! (Valid Loss: {valid_loss:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"  >> İyileşme yok ({epochs_without_improvement}/{EARLY_STOP_PATIENCE})")
        
        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(f"\nErken durdurma! {EARLY_STOP_PATIENCE} epoch boyunca iyileşme olmadı.")
            break

# En iyi modeli geri yükle
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nEn iyi model geri yüklendi (Valid Loss: {best_valid_loss:.4f})")

print("Eğitim tamamlandı!")

In [ ]:
import matplotlib.pyplot as plt

# ============================================================
# Eğitim Sonuçlarını Görselleştir
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss grafiği
ax1.plot(train_losses, label='Train Loss', color='blue')
ax1.plot(valid_losses, label='Valid Loss', color='red')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training vs Validation Loss')
ax1.legend()
ax1.grid(True)

# Accuracy grafiği
ax2.plot(train_accuracies, label='Train Accuracy', color='blue')
ax2.plot(valid_accuracies, label='Valid Accuracy', color='red')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training vs Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nEn iyi Validation Loss: {best_valid_loss:.4f}")
print(f"Son Validation Accuracy: {valid_accuracies[-1]:.2f}%")

In [ ]:
# ============================================================
# Modeli kaydet
# ============================================================
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'class_names': class_names,
    'train_losses': train_losses,
    'valid_losses': valid_losses,
    'train_accuracies': train_accuracies,
    'valid_accuracies': valid_accuracies,
    'best_valid_loss': best_valid_loss,
}, 'baseline_cnn_best.pth')

print("Model 'baseline_cnn_best.pth' olarak kaydedildi!")